In [2]:
# 1C

from pathlib import Path
import copy
import math
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from sklearn.linear_model import LinearRegression

torch.manual_seed(42)
torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)

folder = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
data_path = folder / "taxi_trips.parquet"

if not data_path.is_file():
    from tkinter import Tk, filedialog

    root = Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    selected = filedialog.askopenfilename(
        parent=root,
        title="Select taxi_trips.parquet",
        filetypes=[("Parquet files", "*.parquet")]
    )
    root.destroy()

    if not selected:
        raise FileNotFoundError("No data file selected.")

    data_path = Path(selected)

features = [
    "pickup_longitude", "pickup_latitude",
    "dropoff_longitude", "dropoff_latitude",
    "min_of_day", "day_of_week", "day_of_year"
]

raw = pd.read_parquet(data_path)
df = raw.drop_duplicates()

X = df[features].to_numpy(dtype=np.float32)
y = df["trip_duration"].to_numpy(dtype=np.float64)

assert np.isfinite(X).all() and np.isfinite(y).all()
assert (y > 0).all()

indices = np.random.default_rng(42).permutation(len(df))
n_train, n_val = int(0.70 * len(df)), int(0.15 * len(df))
train_idx, val_idx, test_idx = np.split(
    indices, [n_train, n_train + n_val]
)

y_mean = float(y[train_idx].mean())
y_scale = float(y[train_idx].std(ddof=0))

X_train = torch.from_numpy(X[train_idx])
X_val = torch.from_numpy(X[val_idx])
y_train = torch.from_numpy(
    ((y[train_idx] - y_mean) / y_scale).astype(np.float32)
)
y_val = (y[val_idx] - y_mean) / y_scale


class GaussianMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(7, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 2)
        )
        nn.init.normal_(self.net[-1].weight, std=0.01)
        with torch.no_grad():
            self.net[-1].bias.copy_(torch.tensor([
                0.0, math.log(math.expm1(1.0 - 1e-3))
            ]))

    def forward(self, x):
        output = self.net(x)
        return output[:, 0], F.softplus(output[:, 1]) + 1e-3


model = GaussianMLP()
optimizer = torch.optim.Adam(
    model.parameters(), lr=1e-3, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=4
)


def nll(y_true, mu, sigma):
    return float(np.mean(
        0.5 * np.log(2 * np.pi)
        + np.log(sigma)
        + 0.5 * ((y_true - mu) / sigma) ** 2
    ))


@torch.no_grad()
def predict(x):
    model.eval()
    means, sds = [], []
    for batch in x.split(16384):
        mu, sigma = model(batch)
        means.append(mu.numpy())
        sds.append(sigma.numpy())
    return (
        np.concatenate(means).astype(np.float64),
        np.concatenate(sds).astype(np.float64)
    )


best_val, stale = float("inf"), 0

for epoch in range(1, 81):
    model.train()
    order = torch.randperm(len(X_train))

    for batch in order.split(2048):
        mu, sigma = model(X_train[batch])
        loss = F.gaussian_nll_loss(
            mu, y_train[batch], sigma.square(),
            full=True, eps=1e-12
        )

        if not torch.isfinite(loss):
            raise FloatingPointError("Non-finite training loss.")

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

    val_mu, val_sigma = predict(X_val)
    val_loss = nll(y_val, val_mu, val_sigma) + math.log(y_scale)
    scheduler.step(val_loss)

    if val_loss < best_val:
        best_val = val_loss
        best_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch
        stale = 0
    else:
        stale += 1

    if stale >= 12:
        break

model.load_state_dict(best_state)

linear = LinearRegression().fit(
    X[train_idx].astype(np.float64), y[train_idx]
)
residuals = y[train_idx] - linear.predict(
    X[train_idx].astype(np.float64)
)
linear_sd = float(np.sqrt(np.mean(residuals ** 2)))

y_test = y[test_idx]
mu_scaled, sigma_scaled = predict(torch.from_numpy(X[test_idx]))
mlp_mu = y_mean + y_scale * mu_scaled
mlp_sigma = y_scale * sigma_scaled

test_predictions = {
    "Constant Gaussian": (
        np.full(len(y_test), y_mean),
        np.full(len(y_test), y_scale)
    ),
    "Linear Gaussian": (
        linear.predict(X[test_idx].astype(np.float64)),
        np.full(len(y_test), linear_sd)
    ),
    "Gaussian MLP": (mlp_mu, mlp_sigma)
}

results = pd.DataFrame([
    {
        "Model": name,
        "Test NLL": nll(y_test, mu, sigma),
        "Test RMSE (min)": np.sqrt(np.mean((y_test - mu) ** 2))
    }
    for name, (mu, sigma) in test_predictions.items()
])

print("Duplicates removed:", len(raw) - len(df))
print("Train / validation / test:",
      len(train_idx), len(val_idx), len(test_idx))
print("Selected epoch:", best_epoch)
print(f"Best validation NLL: {best_val:.6f}")
print(results.to_string(index=False, float_format=lambda v: f"{v:.6f}"))

Duplicates removed: 4
Train / validation / test: 366485 78532 78533
Selected epoch: 40
Best validation NLL: 2.677170
            Model  Test NLL  Test RMSE (min)
Constant Gaussian  3.479301         7.846618
  Linear Gaussian  3.449125         7.614007
     Gaussian MLP  2.647189         4.027476


In [3]:
# 1E

from scipy.special import ndtr
from sklearn.metrics import brier_score_loss, accuracy_score

p_test = ndtr((45.0 - mlp_mu) / mlp_sigma)
z_test = (y_test < 45.0).astype(int)
predicted_class = (p_test >= 0.5).astype(int)

print(f"Brier score: {brier_score_loss(z_test, p_test):.8f}")
print(f"Accuracy: {accuracy_score(z_test, predicted_class):.4%}")

Brier score: 0.00303731
Accuracy: 99.6396%
